# 02 – Cleaning, feature engineering and data split

**ML Project – Early Detection of Students at Risk of Dropout** · Week 2

Module owner: **Léo Chen** · Group **AALT** (DIA4): Léo Chen, Adam Gaabi, Timéo Brajon, Axel Warot

In [1]:
import sys,warnings
warnings.filterwarnings("ignore")
sys.path.insert(0,"../src")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from features import *
pd.set_option("display.width",160)
pd.set_option("display.max_columns",40)
from sklearn.model_selection import train_test_split

## 1. Clean dataset
Columns renamed to snake_case, hidden whitespace removed, two engineered features per semester (stateless, row by row, so no leakage).

In [2]:
d=add_features(load_raw("../data/raw/data.csv"))
for c in NOMINAL:d[c]=d[c].astype(int)
d[TARGET]=pd.Categorical(d[TARGET],categories=CLASSES)
assert d.isna().sum().sum()==0 and d.duplicated().sum()==0
d.to_csv("../data/processed/students_clean.csv",index=False)
print(d.shape)
d[["sem1_enrolled","sem1_approved","sem1_approval_rate","sem1_grade","sem1_no_grade",TARGET]].head()

(4424, 41)


,sem1_enrolled,sem1_approved,sem1_approval_rate,sem1_grade,sem1_no_grade,target
0,0,0,0.000000,0.000000,1,Dropout
1,6,6,1.000000,14.000000,0,Graduate
2,6,0,0.000000,0.000000,1,Dropout
3,6,6,1.000000,13.428571,0,Graduate
4,6,5,0.833333,12.333333,0,Graduate


## 2. Train / test split
Stratified 80/20 split with a fixed seed. The **test set is sealed**: it is not used for any decision until the final evaluation (week 4).

In [3]:
tr,te=train_test_split(d,test_size=0.2,stratify=d[TARGET],random_state=SEED)
tr.to_csv("../data/processed/train.csv",index=False)
te.to_csv("../data/processed/test.csv",index=False)
pd.DataFrame({"train":tr[TARGET].value_counts(normalize=True),"test":te[TARGET].value_counts(normalize=True),"full":d[TARGET].value_counts(normalize=True)}).round(3).assign(n_train=tr[TARGET].value_counts(),n_test=te[TARGET].value_counts())

,train,test,full,n_train,n_test
target,,,,,
Graduate,0.499,0.499,0.499,1767,442
Dropout,0.321,0.321,0.321,1137,284
Enrolled,0.179,0.180,0.179,635,159


Validation is done by **stratified k-fold cross-validation on the training set** (no fixed validation set: 3,539 rows are too few to lose 20% more).

## 3. Feature scenarios
B and C also include the 2 engineered features of each available semester.

In [4]:
pd.DataFrame([{"scenario":s,"available":w,"raw features":len(columns(s))} for s,w in zip(SCENARIOS,["at enrollment","after semester 1","after semester 2 (upper bound)"])])

,scenario,available,raw features
0,A_enrollment,at enrollment,24
1,B_semester1,after semester 1,32
2,C_all,after semester 2 (upper bound),40


## 4. Preprocessing pipeline
One-hot encoding for nominal codes (rare categories grouped as *infrequent*), passthrough for binary flags, standardisation for numeric features. It is always fitted on training data only.

In [5]:
for s in SCENARIOS:
    X,y=split_xy(tr,s)
    p=preprocessor(s).fit(X)
    print(f"{s:14s} raw = {X.shape[1]:2d} -> encoded = {p.transform(X).shape[1]}")
p=preprocessor("A_enrollment").fit(split_xy(tr,"A_enrollment")[0])
oh=p.named_transformers_["nom"]
pd.DataFrame({"feature":NOMINAL,"categories":[len(c) for c in oh.categories_],"kept after grouping":[len(c)-(len(i) if i is not None else 0)+(1 if i is not None else 0) for c,i in zip(oh.categories_,oh.infrequent_categories_)]})

A_enrollment   raw = 24 -> encoded = 107
B_semester1    raw = 32 -> encoded = 115
C_all          raw = 40 -> encoded = 123


,feature,categories,kept after grouping
0,marital_status,6,4
1,application_mode,18,13
2,course,17,17
3,previous_qualification,17,9
4,nationality,19,3
5,mother_qualification,28,10
6,father_qualification,32,10
7,mother_occupation,31,13
8,father_occupation,41,13


## Leakage guards
- Split done once, before any fitted transformation; test set never read in notebooks 03–04.
- Encoder, scaler, feature selection and class weights live inside one `Pipeline`, refitted on each training fold.
- Semester 2 features are excluded from scenarios A and B; semester 1 features from scenario A.
- Engineered features only use values of the same row.